# Architecture Upgrade Chain: Models F → G → H

This notebook covers the full pipeline for the three upgraded VQA models:

| Model | Question Encoder | Decoder | New Component |
|-------|-----------------|---------|---------------|
| **E** | BiLSTM (scratch) | LSTM + Bahdanau | ← baseline |
| **F** | **CLIP Text Transformer** | LSTM + Bahdanau | CLIP text encoder |
| **G** | CLIP Text Transformer | LSTM + **MHA (8-head)** | Multi-Head cross-attention |
| **H** | CLIP Text Transformer | **Transformer Decoder** | Full pre-norm Transformer |

Each model adds **one** improvement — making the contribution of each upgrade directly measurable.

---
**Hardware:** RTX 5070 Ti (16 GB VRAM) · i7-14700KF · BFloat16 AMP

## 0 · Setup

In [ ]:
!nvidia-smi

In [ ]:
import os, sys, json, random
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

sys.path.insert(0, 'src')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability()
    print(f'Compute : {cap[0]}.{cap[1]}  ({"BFloat16 supported" if cap[0] >= 8 else "FP16 only"})')

In [ ]:
from vocab import Vocabulary

vocab_q = Vocabulary()
vocab_q.load('data/processed/vocab_questions.json')

vocab_a = Vocabulary()
vocab_a.load('data/processed/vocab_answers.json')

print(f'vocab_q : {len(vocab_q):,} tokens')
print(f'vocab_a : {len(vocab_a):,} tokens')
print(f'Special : pad={vocab_a.pad_idx} start={vocab_a.start_idx} end={vocab_a.end_idx}')

---
# Model F — CLIP Text Encoder

**Upgrade over E**: BiLSTM → CLIP Text Transformer

CLIP was trained with InfoNCE on 400M image-text pairs, so `img_feat · text_feat` is already a meaningful cross-modal similarity signal *before* any VQA fine-tuning. The FiLM generator starts from a pre-aligned space instead of random BiLSTM weights trained only on ~200K VQA-E samples.

**Architecture diff**:
```
E: questions (B, Q)  → BiLSTM → q_feat (B, 1024)
F: clip_ids  (B, 77) → CLIP Text Transformer → q_feat (B, 1024)
```
Everything else (FiLMFusion, LSTMDecoder + Bahdanau) is unchanged.

## F · Phase 1 — Base Training (15 epochs)

Freeze both CLIP backbones (vision + text). Train only FiLMFusion, projections, and decoder.

In [ ]:
!python src/train.py --model F \
    --epochs 15 \
    --batch_size 128 \
    --num_workers 12 \
    --lr 1e-3 \
    --warmup_epochs 3 \
    --augment

## F · Phase 2 — Fine-tune CLIP backbones (10 epochs)

Unfreeze the top layers of both CLIP ViT (image) and CLIP Text encoders.  
Differential LR: backbone params get `1e-5` (0.1× base), rest get `1e-4`.

In [ ]:
!python src/train.py --model F \
    --epochs 10 \
    --batch_size 128 \
    --num_workers 12 \
    --lr 1e-4 \
    --warmup_epochs 0 \
    --resume checkpoints/model_f_best.pth \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --coverage \
    --coverage_lambda 1.0

## F · Phase 3 — Scheduled Sampling (5 epochs)

Reduce exposure bias: decoder gradually receives its own predictions instead of always GT tokens.

In [ ]:
!python src/train.py --model F \
    --epochs 5 \
    --batch_size 128 \
    --num_workers 12 \
    --lr 5e-5 \
    --warmup_epochs 0 \
    --resume checkpoints/model_f_best.pth \
    --scheduled_sampling \
    --ss_k 5.0 \
    --coverage \
    --coverage_lambda 1.0

## F · Phase 4 — RL / SCST (3 epochs, optional)

Self-Critical Sequence Training directly optimizes BLEU-4 via REINFORCE.

In [ ]:
!python src/train_rl.py \
    --model_type F \
    --base_checkpoint checkpoints/model_f_best.pth \
    --epochs 3 \
    --batch_size 64 \
    --lr 1e-5

---
# Model G — Multi-Head Cross-Attention

**Upgrade over F**: Single-head Bahdanau → 8-head scaled dot-product attention

Bahdanau uses `tanh` which saturates gradients. MHA uses `softmax(QKᵀ / √d_k)` — the `1/√d_k` scaling keeps dot-product variance at 1.0, preventing saturation.

With 8 heads (`d_k = 1024/8 = 128`), the decoder can simultaneously attend to 8 different question intents per step (location, colour, cardinality, binary yes/no, ...).

**Architecture diff**:
```
F: BahdanauAttention(hidden=1024, attn_dim=512)    # 1 head
G: nn.MultiheadAttention(embed_dim=1024, heads=8)  # 8 heads
```

## G · Phase 1 — Base Training (15 epochs)

In [ ]:
!python src/train.py --model G \
    --epochs 15 \
    --batch_size 128 \
    --num_workers 12 \
    --lr 1e-3 \
    --warmup_epochs 3 \
    --augment

## G · Phase 2 — Fine-tune CLIP backbones (10 epochs)

In [ ]:
!python src/train.py --model G \
    --epochs 10 \
    --batch_size 128 \
    --num_workers 12 \
    --lr 1e-4 \
    --warmup_epochs 0 \
    --resume checkpoints/model_g_best.pth \
    --finetune_cnn \
    --cnn_lr_factor 0.1

## G · Phase 3 — Scheduled Sampling (5 epochs)

In [ ]:
!python src/train.py --model G \
    --epochs 5 \
    --batch_size 128 \
    --num_workers 12 \
    --lr 5e-5 \
    --warmup_epochs 0 \
    --resume checkpoints/model_g_best.pth \
    --scheduled_sampling \
    --ss_k 5.0

## G · Phase 4 — RL / SCST (3 epochs, optional)

In [ ]:
!python src/train_rl.py \
    --model_type G \
    --base_checkpoint checkpoints/model_g_best.pth \
    --epochs 3 \
    --batch_size 64 \
    --lr 1e-5

---
# Model H — Full Transformer Decoder

**Upgrade over G**: LSTM decoder → Pre-norm Transformer decoder (4 layers × 8 heads)

The LSTM compresses all past token context into a 1024-d vector `h_{t-1}`. For 15–25 token explanations this is a hard compression bottleneck. The Transformer keeps every previous token explicit and attends over them directly:

```
# At step t — O(1) gradient path to any past token
self_attn  = MaskedMHA(y_t, [y_0 .. y_{t-1}])   # history
cross_img  = MHA(y_t, img_patches)               # 49 visual regions  
cross_q    = MHA(y_t, q_tokens)                  # 77 language tokens
y_t_out    = FFN(LayerNorm(self_attn + cross_img + cross_q))
```

**Training**: all T positions processed in **parallel** via causal mask — no sequential bottleneck.  
**Inference**: KV-cache (accumulated embedding buffer grows by 1 each step).

> Note: Scheduled Sampling (Phase 3) falls back to teacher-forcing for H — the Transformer already processes all tokens in parallel, making token-by-token SS inapplicable.

## H · Phase 1 — Base Training (15 epochs)

Lower dropout (0.1 vs 0.5) since Transformer uses attention dropout internally.

In [ ]:
!python src/train.py --model H \
    --epochs 15 \
    --batch_size 64 \
    --num_workers 12 \
    --lr 5e-4 \
    --warmup_epochs 5 \
    --augment \
    --label_smoothing 0.1

## H · Phase 2 — Fine-tune CLIP backbones (10 epochs)

Longer warmup not needed — Transformer is already well-calibrated at this point.

In [ ]:
!python src/train.py --model H \
    --epochs 10 \
    --batch_size 64 \
    --num_workers 12 \
    --lr 1e-4 \
    --warmup_epochs 0 \
    --resume checkpoints/model_h_best.pth \
    --finetune_cnn \
    --cnn_lr_factor 0.1

## H · Phase 3 — Scheduled Sampling / continued training (5 epochs)

For Model H, `--scheduled_sampling` falls back to teacher-forcing (Transformer trains in parallel). This phase is effectively continued fine-tuning at a lower LR.

In [ ]:
!python src/train.py --model H \
    --epochs 5 \
    --batch_size 64 \
    --num_workers 12 \
    --lr 2e-5 \
    --warmup_epochs 0 \
    --resume checkpoints/model_h_best.pth \
    --scheduled_sampling \
    --ss_k 5.0

## H · Phase 4 — RL / SCST (3 epochs, optional)

Transformer policies tend to be sharper (lower entropy) than LSTM policies, which means lower REINFORCE variance. SCST often gives a larger BLEU gain for Transformer decoders.

In [ ]:
!python src/train_rl.py \
    --model_type H \
    --base_checkpoint checkpoints/model_h_best.pth \
    --epochs 3 \
    --batch_size 32 \
    --lr 1e-5

---
# Evaluation

Run NLP metrics (BLEU-1/2/3/4, METEOR, Exact Match) on the validation set.

In [ ]:
print('=' * 60)
print('Model F — Greedy')
print('=' * 60)
!python src/evaluate.py --model_type F --checkpoint checkpoints/model_f_best.pth

In [ ]:
print('=' * 60)
print('Model F — Beam Search (width=3)')
print('=' * 60)
!python src/evaluate.py --model_type F --checkpoint checkpoints/model_f_best.pth \
    --beam_width 3 --no_repeat_ngram 3

In [ ]:
print('=' * 60)
print('Model G — Greedy')
print('=' * 60)
!python src/evaluate.py --model_type G --checkpoint checkpoints/model_g_best.pth

In [ ]:
print('=' * 60)
print('Model G — Beam Search (width=3)')
print('=' * 60)
!python src/evaluate.py --model_type G --checkpoint checkpoints/model_g_best.pth \
    --beam_width 3 --no_repeat_ngram 3

In [ ]:
print('=' * 60)
print('Model H — Greedy')
print('=' * 60)
!python src/evaluate.py --model_type H --checkpoint checkpoints/model_h_best.pth

In [ ]:
print('=' * 60)
print('Model H — Beam Search (width=3)')
print('=' * 60)
!python src/evaluate.py --model_type H --checkpoint checkpoints/model_h_best.pth \
    --beam_width 3 --no_repeat_ngram 3

## Compare E → F → G → H

Side-by-side BLEU/METEOR table for all models.

In [ ]:
!python src/compare.py --models E,F,G,H --num_samples 2000 --beam_width 1

## Loss Curves

In [ ]:
import json, os
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'F': '#2ca02c', 'G': '#ff7f0e', 'H': '#d62728', 'E': '#1f77b4'}

for ax, model in zip(axes, ['F', 'G', 'H']):
    path = f'checkpoints/history_model_{model.lower()}.json'
    if not os.path.exists(path):
        ax.set_title(f'Model {model} — no history yet')
        continue
    with open(path) as f:
        h = json.load(f)
    ax.plot(h['train_loss'], label='Train', color=colors[model], linewidth=2)
    ax.plot(h['val_loss'],   label='Val',   color=colors[model], linewidth=2, linestyle='--')

    # Overlay Model E as baseline reference
    e_path = 'checkpoints/history_model_e.json'
    if os.path.exists(e_path):
        with open(e_path) as f:
            he = json.load(f)
        n = len(h['val_loss'])
        ax.plot(he['val_loss'][:n], label='E (baseline)',
                color=colors['E'], linewidth=1.5, linestyle=':', alpha=0.7)

    ax.set_title(f'Model {model}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training Curves — F / G / H vs E baseline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('checkpoints/curves_fgh.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: checkpoints/curves_fgh.png')

---
# Inference

Single-sample greedy and beam decode for all three new models.

In [ ]:
from inference import (
    load_model_from_checkpoint,
    greedy_decode_with_attention,
    beam_search_decode_with_attention,
)
from transformers import CLIPProcessor

# Image transform (same as training)
_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# CLIP processor for F/G/H question tokenisation
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

def tokenise_question_clip(question: str):
    """Return LongTensor (77,) of CLIP BPE token ids."""
    enc = clip_processor(
        text=question, return_tensors='pt',
        padding='max_length', max_length=77, truncation=True,
    )
    return enc['input_ids'].squeeze(0)   # (77,)

print('Inference helpers ready.')

In [ ]:
loaded_models = {}

for name in ['F', 'G', 'H']:
    ckpt = f'checkpoints/model_{name.lower()}_best.pth'
    if os.path.exists(ckpt):
        loaded_models[name] = load_model_from_checkpoint(
            name, ckpt, len(vocab_q), len(vocab_a), device=DEVICE
        )
        print(f'Loaded Model {name}')
    else:
        print(f'[SKIP] {ckpt} not found — train first')

# Also load Model E as baseline if available
e_ckpt = 'checkpoints/model_e_best.pth'
if os.path.exists(e_ckpt):
    from models.vqa_models import VQAModelE
    loaded_models['E'] = load_model_from_checkpoint(
        'E', e_ckpt, len(vocab_q), len(vocab_a), device=DEVICE
    )
    print('Loaded Model E (baseline)')

In [ ]:
# ── Pick a VQA-E sample ───────────────────────────────────────────────────────
VQA_JSON   = 'data/vqa_e/VQA-E_val_set.json'
IMAGE_DIR  = 'data/raw/val2014'
SAMPLE_IDX = 0    # change to try different samples

with open(VQA_JSON) as f:
    annotations = json.load(f)

sample      = annotations[SAMPLE_IDX]
q_text      = sample['question']
img_id      = sample['img_id']
answer_gt   = sample.get('multiple_choice_answer', '')
explanation = sample.get('explanation', [''])[0]

img_path    = os.path.join(IMAGE_DIR, f'COCO_val2014_{img_id:012d}.jpg')
img_pil     = Image.open(img_path).convert('RGB')
img_tensor  = _TRANSFORM(img_pil)

# Tokenise for CLIP models
clip_ids    = tokenise_question_clip(q_text)
# Tokenise for Model E (vocab_q)
q_tensor_e  = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

plt.figure(figsize=(5, 4))
plt.imshow(img_pil)
plt.axis('off')
plt.title(f'Q: {q_text}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Question   : {q_text}')
print(f'GT Answer  : {answer_gt}')
print(f'GT Explain : {explanation}')

In [ ]:
# ── Decode with all loaded models ─────────────────────────────────────────────
print(f'Question: {q_text}')
print(f'GT      : {answer_gt} because {explanation}')
print('=' * 70)

for name, model in sorted(loaded_models.items()):
    q_in = q_tensor_e if name == 'E' else clip_ids

    with torch.no_grad():
        greedy = greedy_decode_with_attention(
            model, img_tensor, q_in, vocab_a, max_len=60, device=DEVICE
        )
        beam = beam_search_decode_with_attention(
            model, img_tensor, q_in, vocab_a,
            beam_width=3, max_len=60, device=DEVICE
        )

    print(f'\nModel {name}')
    print(f'  Greedy : {greedy}')
    print(f'  Beam-3 : {beam}')

## Batch Inference — multiple samples

In [ ]:
from inference import batch_greedy_decode_with_attention
from torch.utils.data import DataLoader
from dataset_clip import VQAEDatasetCLIP, clip_collate_fn

N_SAMPLES = 20
MODEL_TO_TEST = 'H'   # change to 'F' or 'G'

if MODEL_TO_TEST not in loaded_models:
    print(f'Model {MODEL_TO_TEST} not loaded — train first.')
else:
    ds = VQAEDatasetCLIP(
        image_dir=IMAGE_DIR,
        vqa_e_json_path=VQA_JSON,
        vocab_a=vocab_a,
        split='val2014',
        max_samples=N_SAMPLES,
    )
    loader = DataLoader(ds, batch_size=N_SAMPLES, collate_fn=clip_collate_fn)
    imgs_b, clip_ids_b, answers_b = next(iter(loader))

    model = loaded_models[MODEL_TO_TEST]
    with torch.no_grad():
        preds = batch_greedy_decode_with_attention(
            model, imgs_b, clip_ids_b, vocab_a, max_len=60, device=DEVICE
        )

    skip = {vocab_a.pad_idx, vocab_a.start_idx, vocab_a.end_idx}
    for i, (pred, a_t) in enumerate(zip(preds, answers_b)):
        gt = ' '.join(vocab_a.idx2word[int(x)] for x in a_t if int(x) not in skip)
        q  = annotations[i]['question']
        print(f'[{i:2d}] Q  : {q}')
        print(f'     GT : {gt}')
        print(f'     Pr : {pred}')
        print()

## Attention Heatmap Visualisation

Visualise per-token image attention weights for Models F, G, H.

> Note: `visualize.py` currently supports C/D/E only (uses `vocab_q` tokenisation).  
> The cell below implements a CLIP-compatible version inline.

In [ ]:
import numpy as np
import torch.nn.functional as F

def visualize_clip_attention(
    model, img_tensor, clip_input_ids, vocab_a,
    model_name='H', max_len=25, device=DEVICE,
    output_path=None,
):
    """Greedy decode + per-token image attention heatmap for F/G/H models."""
    from inference import _encode_spatial, _is_transformer

    model.eval()
    with torch.no_grad():
        imgs = img_tensor.unsqueeze(0).to(device)
        qids = clip_input_ids.unsqueeze(0).to(device)

        enc_hidden, spatial_feats, q_hidden = _encode_spatial(model, imgs, qids, device)
        is_tx = _is_transformer(model.decoder)

        token     = torch.tensor([[vocab_a.start_idx]], dtype=torch.long, device=device)
        hidden    = None if is_tx else enc_hidden
        coverage  = None

        tokens_list = []
        alphas_list = []

        for _ in range(max_len):
            if is_tx:
                logit, hidden, alpha, _ = model.decoder.decode_step(
                    token, hidden, spatial_feats, q_hidden
                )
            else:
                logit, hidden, alpha, coverage = model.decoder.decode_step(
                    token, hidden, spatial_feats, q_hidden, coverage
                )

            pred = logit.argmax(dim=-1).item()
            if pred == vocab_a.end_idx:
                break

            word = vocab_a.idx2word.get(pred, '<unk>')
            tokens_list.append(word)
            alphas_list.append(alpha.squeeze(0).cpu().numpy())  # (49,)
            token = torch.tensor([[pred]], dtype=torch.long, device=device)

    if not alphas_list:
        print('No tokens decoded.')
        return

    # ── Denormalize image ────────────────────────────────────────────────────
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_np = (img_tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    n_cols = len(tokens_list) + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5))

    axes[0].imshow(img_np)
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')

    for i, (word, alpha) in enumerate(zip(tokens_list, alphas_list)):
        attn = alpha.reshape(7, 7)
        attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
        attn_up = np.array(
            Image.fromarray((attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
        ) / 255.0
        axes[i + 1].imshow(img_np)
        axes[i + 1].imshow(attn_up, alpha=0.5, cmap='jet')
        axes[i + 1].set_title(f'"{word}"', fontsize=9)
        axes[i + 1].axis('off')

    fig.suptitle(
        f'Model {model_name} | Q: {q_text}\nA: {" ".join(tokens_list)}',
        fontsize=10, fontweight='bold'
    )
    plt.tight_layout()

    save_path = output_path or f'checkpoints/attn_model_{model_name.lower()}.png'
    os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


# ── Run for each loaded model ─────────────────────────────────────────────────
for name in ['F', 'G', 'H']:
    if name in loaded_models:
        print(f'\n=== Model {name} Attention ===')
        visualize_clip_attention(
            loaded_models[name], img_tensor, clip_ids, vocab_a,
            model_name=name, max_len=20,
        )
    else:
        print(f'[SKIP] Model {name} not loaded.')

---
# Interactive Session

Pick any image and ask questions to all loaded models.

In [ ]:
from interactive import InteractiveVQA

ivqa = InteractiveVQA(
    models=loaded_models,
    vocab_q=vocab_q,
    vocab_a=vocab_a,
    image_dir=IMAGE_DIR,
    device=DEVICE,
    beam_width=3,
    no_repeat_ngram=3,
)

ivqa.pick_image()   # random image from val2014

In [ ]:
ivqa.ask("What is the person doing?")   # change question as needed

---
# LLM-as-Judge Evaluation (Gemini)

Rate model outputs on a 0–5 scale using Gemini as an independent judge.

In [ ]:
# Set your Gemini API key before running
# import os; os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_HERE'

!python src/llm_eval.py \
    --model_type F \
    --checkpoint checkpoints/model_f_best.pth \
    --num_samples 100

---
# Results Summary

Print a formatted table after all models have been evaluated.

In [ ]:
import json, os

# ── Load eval results if saved separately ────────────────────────────────────
# evaluate.py prints metrics but doesn't save to JSON by default.
# Run the cells above and read the printed BLEU-4 / METEOR values manually,
# or adapt evaluate.py to write a JSON file and read it here.

# ── Loss curve final values ───────────────────────────────────────────────────
print(f'{"Model":<8} {"Final Train":<14} {"Final Val":<12} {"Best Val"}')
print('-' * 50)
for m in ['E', 'F', 'G', 'H']:
    path = f'checkpoints/history_model_{m.lower()}.json'
    if not os.path.exists(path):
        print(f'{m:<8} (no history)')
        continue
    with open(path) as f:
        h = json.load(f)
    train_last = h['train_loss'][-1] if h['train_loss'] else float('nan')
    val_last   = h['val_loss'][-1]   if h['val_loss']   else float('nan')
    val_best   = min(h['val_loss'])  if h['val_loss']   else float('nan')
    print(f'{m:<8} {train_last:<14.4f} {val_last:<12.4f} {val_best:.4f}')